# Observer for the bristle MPC

The bristle MPC predicts with states it cannot measure: the chassis velocity and the bristle deflections $z$ (the bristle defines the friction on each wheel). This notebook builds an **observer** that estimates them from what the robot can measure.

An observer runs inside the controller, next to the robot. Every update it:

1. **predicts** its estimate one step ahead with the model,
2. **compares** the prediction with the sensors,
3. **corrects** the estimate by a gain times the difference.

When the robot behaves like the model, the corrections are small. When something happens that the model did not expect (a wheel breaking loose, a grippier patch of floor), the difference shows up in the sensors and the estimate follows it.

This observer runs **every 1 ms**, not every 20 ms, so it follows what happens during each voltage hold instead of seeing only its end. The MPC still plans every 20 ms, starting from the observer's latest estimate. The plant is unchanged.

In [ ]:
# ===========================================================================
# The robot (plant), loaded from the open-loop model notebook
# ===========================================================================
import json
import numpy as np
from scipy.linalg import expm

PLANT_NOTEBOOK = "RobotSimulationOpenLoopModel_Merged.ipynb"

plant_cells = {c.get("id"): "".join(c["source"]) for c in json.load(open(PLANT_NOTEBOOK, encoding="utf-8"))["cells"]}
exec(plant_cells["constants"])     # simulation_parameters, GRAVITY
exec(plant_cells["plant"])         # OmniMotorState, OmniRobotSimulation

print(f"Loaded the robot from {PLANT_NOTEBOOK}")

## Sensors

What the controller can read (Dana's list), and how often:

| Sensor | Reads | Every |
|---|---|---|
| Wheel encoders | wheel speeds $\omega_{1..4}$ | 1 ms (each plant step) |
| Motor drivers | motor currents $I_{1..4}$ | 1 ms |
| Global camera | pose $[X, Y, \theta]$ | 20 ms (assumed) |
| (the controller itself) | voltages $u$ | known: it set them |

The sensors read the plant's true values for now (no sensor noise). The rates are assumptions, kept as settings.

In [ ]:
# ===========================================================================
# Sensors: what the controller can read from the robot
# ===========================================================================

sensor_parameters = {
    'camera_period': 0.02,     # Time between camera pose readings (s), assumed
}


def read_sensors(state: OmniMotorState, k: int, params: dict, sensors: dict) -> tuple:
    """
    What the controller can measure at plant step k.

    state: the robot's true state (the sensors read from it)
    k: plant step number
    params: simulation_parameters (for the plant step dt)
    sensors: sensor_parameters

    Returns (omega, I, pose):
        omega: wheel speeds from the encoders [rad/s]
        I: motor currents from the drivers [A]
        pose: [X, Y, theta] from the camera [m, m, rad], or None between camera readings
    """
    omega = state.omega.copy() # encoders every 1ms (fast moving)
    I = state.I.copy() # motor driver currents every 1ms (fast moving)

    # The camera gives a pose only every camera_period
    camera_every = int(round(sensors['camera_period'] / params['dt']))
    if k % camera_every == 0: # camera reading every 20ms
        pose = state.pos_global.copy()
    else:
        pose = None # no camera reading in  between

    return omega, I, pose

## The observer's model

The bristle model of the MPC, with one change: it is **driven by the measured motor currents**, not the voltages. The current gives the motor torque directly ($K_t I$), so when a wheel speeds up differently from the model, the difference can only come from the friction. That is what lets the observer work out $z$.

**State:** $\mathbf{x} = [X, Y, \theta, v_x, v_y, \dot{\theta}, \omega_{1..4}, z_{1..4}]$: the pose, the velocity along the robot's own axes, the wheel speeds and the bristle deflections. **Input:** the measured currents $\mathbf{I}$.

**Bristle deflection: 100 µm** (the plant uses 2 µm), because the observer works in 1 ms steps. The plant's 2 µm bristle changes much faster than 1 ms: while sliding it settles in microseconds, and while gripping it rings at about 600–800 Hz. Predicting it over a whole 1 ms step is therefore less accurate (the plant itself steps it with backward Euler, which smooths those fast changes). The softer 100 µm bristle changes on about the time scale of the step, so a 1 ms prediction follows the plant more closely.

In a check against the plant, predicting 1 ms ahead with the 100 µm bristle came about five times closer to the plant's bristle state than with 2 µm.

It is also the MPC model's value, so the observer's $z$ is the same state the MPC predicts with.

**Equations** (from the plant, Steps 5.3 to 5.9):

* Drive slip: $v_{s,i} = (\mathbf{H}_{drive}\mathbf{v})_i - r\,\omega_i$
* Bristles: $\delta\, \dot{z}_i = v_{s,i} - d_i z_i$
* Friction: $F_i = -\mu_s F_{n,i}\, z_i$
* Wheels: $J_w \dot{\omega}_i = K_t I_i - c_i F_{n,i}\, r - r F_i$ (no back-EMF term: the current is measured)
* Chassis: $\frac{d\mathbf{v}}{dt} = \mathbf{M}^{-1}\left(\mathbf{H}_{drive}^\top \mathbf{F} + \mathbf{H}_{axial}^\top (\boldsymbol{\phi}_a \circ \mathbf{F}_n)\right) + \dot{\theta}\,[v_y,\ -v_x,\ 0]$. The last term is the turning of the robot's own axes: $\mathbf{v}$ is measured along axes that rotate at $\dot{\theta}$, so it changes direction even with no force (it comes from differentiating $\mathbf{v} = \mathbf{R}(\theta)^\top \mathbf{\dot{X}}_g$).
* Pose: $\frac{d}{dt}[X, Y, \theta] = \mathbf{R}(\theta)\,\mathbf{v}$

**Held for one observer step (1 ms):** the relax rate $d_i$, the normal forces $F_{n,i}$, the rolling drag $c_i$, the roller force $\phi_{a,i}$ and the turning rate, worked out at the current estimate. Over 1 ms they barely change, and they are worked out again every step. With them held, the velocity, wheel speed and bristle equations are linear and are stepped exactly over 1 ms with one matrix exponential (the bristles are too fast for forward Euler, even over 1 ms). The pose moves by $\Delta T\, \mathbf{R}(\theta)\mathbf{v}$.

In [ ]:
"""
The observer's model
====================

ObserverModel is the bristle model stepped every 1 ms and driven by the measured motor currents.

Calculates, for one observer step dt:
    - the next state estimate, from the current estimate and the measured currents
    - the transition matrix Phi (how a small error in the estimate grows over the step), for the observer's gain
"""


class ObserverModel:
    """
    State x = [X, Y, theta, v_x, v_y, theta_dot (along the robot's axes), omega_1..4, z_1..4]; input I = measured currents.
    """

    # Where each part of the state sits in x
    POSE = slice(0, 3)          # [X, Y, theta]
    #we dont have this. integrator.. VELOCITY = slice(3, 6)      # v = [v_x, v_y, theta_dot], along the robot's axes
    WHEELS = slice(6, 10)       # omega_1..4
    BRISTLES = slice(10, 14)    # z_1..4

    def __init__(self, params: dict, deflection: float = 100e-6):
        """
        params: simulation_parameters (the same values as the plant)
        deflection: the model's bristle deflection delta [m] (the plant uses 2 um; the MPC model uses 100 um)
        """
        self.p = params
        self.delta = deflection

        # -------------------------------------------------------------------
        # Fixed geometry and loads, worked out by the plant's robot class from the same parameters.
        # Only fixed values are read from it, never a state.
        # -------------------------------------------------------------------
        geometry = OmniRobotSimulation(params)

        # Drive speeds of the wheels = H_drive v, axle (roller) speeds = H_axial v
        self.H_drive = geometry.H_drive
        self.H_axial = geometry.H_axial

        # 1 / mass and 1 / yaw inertia, for [x, y, yaw]
        self.M_inv = np.diag(geometry.M_inv)

        # Normal force on each wheel at rest [N]
        self.F_n_static = np.array([geometry.F_n_front_static, geometry.F_n_front_static,
                                    geometry.F_n_rear_static, geometry.F_n_rear_static])

        # Load transfer (Section 4) as a matrix: F_n = F_n_static + load_transfer a, with a = [right, forward, yaw]
        pitch = geometry.pitch_gain
        roll_front = geometry.roll_gain_front
        roll_rear = geometry.roll_gain_rear
        self.load_transfer = np.array([
            [-roll_front, -pitch, 0.0],     # 1 FR
            [roll_front, -pitch, 0.0],      # 2 FL
            [roll_rear, pitch, 0.0],        # 3 RL
            [-roll_rear, pitch, 0.0],       # 4 RR
        ])

    # =======================================================================
    # Values held for one step
    # =======================================================================

    def held_values(self, x: np.ndarray) -> tuple:
        """
        The values held for one observer step, worked out at the estimate x.

        Returns (d, F_n, c_roll, phi_a, turn):
            d: bristle relax rate of each wheel [m/s]: small while gripping, large while sliding (Step 5.5)
            F_n: normal force on each wheel, with this state's acceleration (Section 4) [N]
            c_roll: rolling drag per unit normal force (Step 5.3)
            phi_a: roller force per unit normal force (Step 5.8)
            turn: turning rate theta_dot [rad/s]
        """
        p = self.p
        v = x[self.VELOCITY]
        omega = x[self.WHEELS]
        z = x[self.BRISTLES]

        # Drive slip of each wheel: v_slip_d = H_drive v - r omega
        v_slip_d = self.H_drive @ v - p['r'] * omega

        # Bristle relax rate (Steps 5.4 and 5.5): d = (mu_s / mu_d) sqrt(v_slip_d^2 + v_c^2)
        mu_d = p['mu_k'] + (p['mu_s'] - p['mu_k']) * np.exp(-p['alpha_d'] * np.abs(v_slip_d))
        d = (p['mu_s'] / mu_d) * np.sqrt(v_slip_d**2 + p['lugre_creep']**2)

        # Roller force per unit normal force, from the axle slip v_slip_a = H_axial v
        if p['roller_friction']:
            v_slip_a = self.H_axial @ v
            mu_a = p['mu_a_k'] + (p['mu_a_s'] - p['mu_a_k']) * np.exp(-p['alpha_a'] * np.abs(v_slip_a))
            phi_a = -mu_a * (2.0 / np.pi) * np.arctan(p['roller_smoothing'] * v_slip_a)
        else:
            phi_a = np.zeros(4)

        # Normal forces with this state's acceleration. The floor forces are a factor times each wheel's normal
        # force, F = -mu_s F_n z and F_a = phi_a F_n, so
        #   a = M^-1 K F_n,   K = -mu_s H_drive^T diag(z) + H_axial^T diag(phi_a)
        # and with F_n = F_n_static + load_transfer a:
        #   (I - load_transfer M^-1 K) F_n = F_n_static
        # A 4 x 4 solve: about 10-20 us, so it is cheap even every 1 ms.
        K = -p['mu_s'] * self.H_drive.T * z + self.H_axial.T * phi_a
        F_n = np.linalg.solve(np.eye(4) - self.load_transfer @ (self.M_inv[:, None] * K), self.F_n_static)
        F_n = np.maximum(0.0, F_n)              # a wheel cannot pull on the floor

        # Rolling drag per unit normal force
        c_roll = p['Crr'] * np.tanh(omega / p['roll_smoothing'])

        turn = v[2]
        return d, F_n, c_roll, phi_a, turn

    # =======================================================================
    # One observer step
    # =======================================================================

    def step(self, x: np.ndarray, I: np.ndarray, dt: float) -> tuple:
        """
        The estimate x moved on by dt, driven by the measured currents I.

        x: state estimate (14,)
        I: measured motor currents [A]
        dt: observer step [s]

        Returns (x_next, Phi):
            x_next: the predicted state after dt
            Phi: transition matrix (14, 14), how a small error in x grows over dt (for the observer's gain)
        """
        p = self.p
        r = p['r']
        Jw = p['Jw']
        mu_s = p['mu_s']
        delta = self.delta
        d, F_n, c_roll, phi_a, turn = self.held_values(x)

        # The state, by name
        pose = x[self.POSE]
        theta = pose[2]
        v = x[self.VELOCITY]
        omega = x[self.WHEELS]
        z = x[self.BRISTLES]

        # -------------------------------------------------------------------
        # Velocity, wheel speeds and bristles, with the values held:
        #   dw/dt = A w + c
        # where w is the velocity v (3 values), the wheel speeds omega (4) and the bristles z (4), stacked into one
        # column of 11 values so the three equations can be stepped together:
        #
        #   [ dv/dt     ]   [ dv_dv   0              dv_dz     ] [ v     ]   [ dv_const     ]
        #   [ domega/dt ] = [ 0       domega_domega  domega_dz ] [ omega ] + [ domega_const ]
        #   [ dz/dt     ]   [ dz_dv   dz_domega      dz_dz     ] [ z     ]   [ 0            ]
        #
        # Each entry is worked out below.
        # -------------------------------------------------------------------

        # Chassis (Step 5.9), along the robot's own axes. The floor forces F = -mu_s F_n z push the chassis,
        # and the roller forces F_a = phi_a F_n add a fixed push:
        #   dv/dt = M^-1 (H_drive^T F + H_axial^T F_a) + axes turning
        # The axes turning: the robot's axes rotate at theta_dot, so a velocity along them changes direction even
        # with no force. Differentiating v = R(theta)^T v_global gives dv_x/dt += theta_dot v_y, dv_y/dt -= theta_dot v_x.
        dv_dv = np.array([[0.0, turn, 0.0],
                          [-turn, 0.0, 0.0],
                          [0.0, 0.0, 0.0]])
        dv_dz = -mu_s * self.M_inv[:, None] * (self.H_drive.T * F_n)       # (3, 4): bristles push the chassis
        dv_const = self.M_inv * (self.H_axial.T @ (phi_a * F_n))          # (3,): roller forces

        # Wheels (Step 5.7), with the measured current. The back-EMF is already in the measured current.
        #   Jw domega/dt = Kt I - c_roll F_n r - r F,   with -r F = r mu_s F_n z
        domega_domega = np.zeros((4, 4))                                   # nothing: the current is measured
        domega_dz = np.diag(r * mu_s * F_n / Jw)                           # (4, 4): bristles hold the wheel back
        domega_const = (p['Kt'] * I - c_roll * F_n * r) / Jw              # (4,): motor torque and rolling drag

        # Bristles (Step 5.5): they stretch with the drive slip and relax at the rate d
        #   delta dz/dt = v_slip_d - d z = H_drive v - r omega - d z
        dz_dv = self.H_drive / delta                                       # (4, 3): chassis motion stretches them
        dz_domega = -r * np.eye(4) / delta                                 # (4, 4): wheel spin stretches them back
        dz_dz = -np.diag(d) / delta                                        # (4, 4): they relax

        # A and c from the entries, laid out as in the equation at the top of this section
        zeros = np.zeros
        A = np.block([
            [dv_dv,         zeros((3, 4)),  dv_dz],
            [zeros((4, 3)), domega_domega,  domega_dz],
            [dz_dv,         dz_domega,      dz_dz],
        ])
        c = np.concatenate((dv_const, domega_const, zeros(4)))

        # Exact step over dt. With A and c held over the step, dw/dt = A w + c has the exact solution
        #   w_next = e^(A dt) w + (integral from 0 to dt of e^(A s) ds) c
        #          =    A_d   w +                 c_d
        # One matrix exponential gives both:
        #   exp([[A, c],    dt)  =  [[A_d, c_d],
        #        [0, 0]]              [0,   1  ]]
        n = 11
        block = np.zeros((n + 1, n + 1))
        block[:n, :n] = A
        block[:n, n] = c
        E = expm(block * dt)
        A_d = E[:n, :n]
        c_d = E[:n, n]

        w = np.concatenate((v, omega, z))          # stack v, omega and z into one column of 11
        w_next = A_d @ w + c_d                     # the exact step

        # The results, by name
        v_next = w_next[0:3]
        omega_next = w_next[3:7]
        z_next = w_next[7:11]

        # -------------------------------------------------------------------
        # Pose: d[X, Y, theta]/dt = R(theta) v, over dt
        # -------------------------------------------------------------------
        R = np.array([[np.cos(theta), -np.sin(theta), 0.0],
                      [np.sin(theta), np.cos(theta),  0.0],
                      [0.0,           0.0,            1.0]])
        pose_next = pose + dt * R @ v

        # The next state
        x_next = np.concatenate((pose_next, v_next, omega_next, z_next))

        # -------------------------------------------------------------------
        # Phi = d(x_next) / d(x): how the next state changes when the state changes a little. The observer
        # (part 4) uses it to carry its uncertainty forward. With x = [pose, w]:
        # 
        #   Phi = [ d(pose_next)/dpose           d(pose_next)/dw         ]
        #         [ d(w_next)/dpose              d(w_next)/dw            ] 
        #
        # w = np.concatenate((v, omega, z)), so dw has parts in dv, domega, dz
        # NOTE: pose_next = pose + dt * R @ v => dpose, and dw only has dv in it.
        # NOTE: w_next    = A_d  @ w + c_d  => no dpose, only dw.
        # pose := p , theta := th, omega := o
        #
        #       = [ d(p + dt * R @ v)/dp                                d(p + dt * R @ v)/dw              ]
        #         [ 0                                                   d(A_d  @ w + c_d)/dw              ]
        #
        #       = [ dp/dx      dp/dy      d(p + dt * dR/dth @ v)/dth    d(dt * R @ v)/dv  d(~)/do  d(~)/dz]
        #       = [ 0          0          0                                               A_d             ]
        #       = [ [1,0,0]^T  [0,1,0]^T  [0,0,1]^T + dt * dR/dth @ v   dt * R            0        0      ]          
        #         [ 0's                                                                   A_d             ]
        #
        # dR/dtheta = R(theta) rot_pos_90deg, the same rotation turned a further +90 degrees:
        #   rot_pos_90deg = [[cos 90, -sin 90, 0],
        #                    [sin 90,  cos 90, 0],
        #                    [0,       0,      0]]
        rot_pos_90deg = np.array([[0.0, -1.0, 0.0],
                                  [1.0,  0.0, 0.0],
                                  [0.0,  0.0, 0.0]])
        dR_dtheta = R @ rot_pos_90deg

        dpose_next_dpose = np.eye(3)                           # d(pose_next)/d(pose): one-for-one
        dpose_next_dpose[:, 2] += dt * dR_dtheta @ v           # ... plus the theta column (column 2 of the pose is theta)
        dpose_next_dv = dt * R                                 # d(pose_next)/d(v)

        Phi = np.block([
            [dpose_next_dpose,   dpose_next_dv,  np.zeros((3, 4)),  np.zeros((3, 4))],
            [np.zeros((11, 3)),  A_d],
        ])

        return x_next, Phi